# 🧠 SafeGuard Micro — TinyML Keyword Spotter for ESP32-C3

### 🎯 Why Micro Keyword Spotting (KWS)?
- Full Speech-to-Text models are 20MB–50MB and **cannot fit in ESP32-C3's 400KB RAM**.
- This notebook trains an ultra-efficient **TinyML DS-CNN (Depthwise Separable CNN)**:
  - **Model Size**: ~25 KB to 40 KB (Quantized INT8)
  - **RAM Usage**: ~18 KB Tensor Arena (easily runs alongside BLE & GPS)
  - **Inference Time**: ~35ms on ESP32-C3 (160MHz RISC-V)
- **Target 2-Word Emergency Classes**:
  1. `_silence_` (Background ambient noise)
  2. `_unknown_` (General speech streamed directly from Mozilla Common Voice)
  3. `tuma_msaada` (Kiswahili: Send help)
  4. `piga_polisi` (Kiswahili: Call police)
  5. `send_help` (English: Send help)
  6. `call_police` (English: Call police)

Directly outputs `safeguard_micro_model.h` for your ESP32-C3 firmware!

In [ ]:
# ===========================================================
# CELL 1 — Setup Environment & Workspace
# ===========================================================
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/SafeGuard-STT"
    print("Mounted Google Drive successfully.")
except Exception as e:
    WORK_DIR = "/content/SafeGuard-STT"
    print(f"Using local Colab directory: {WORK_DIR}")

MICRO_DIR = f"{WORK_DIR}/micro_kws"
os.makedirs(MICRO_DIR, exist_ok=True)
os.makedirs(f"{MICRO_DIR}/dataset", exist_ok=True)

print(f"Micro KWS workspace: {MICRO_DIR}")

In [ ]:
# ===========================================================
# CELL 2 — Install Dependencies
# ===========================================================
!pip install -q tensorflow datasets librosa soundfile numpy matplotlib tqdm gTTS

import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ===========================================================
# CELL 3 — Prepare Dataset (Auto-Synthesizes & Augments if No Recordings)
# ===========================================================
import os, glob, shutil, io, random
import soundfile as sf
import librosa
import numpy as np
from tqdm import tqdm
from gtts import gTTS

CLASSES = ["_silence_", "_unknown_", "tuma_msaada", "piga_polisi", "send_help", "call_police"]
for c in CLASSES:
    os.makedirs(f"{MICRO_DIR}/dataset/{c}", exist_ok=True)

TARGET_CONFIG = [
    ("tuma_msaada", "tuma msaada", "sw"),
    ("piga_polisi", "piga polisi", "sw"),
    ("send_help", "send help", "en"),
    ("call_police", "call police", "en"),
]

def pad_or_trim(arr, target_len=16000):
    if len(arr) < target_len:
        pad_left = (target_len - len(arr)) // 2
        pad_right = target_len - len(arr) - pad_left
        arr = np.pad(arr, (pad_left, pad_right), mode="constant")
    else:
        arr = arr[:target_len]
    max_val = np.max(np.abs(arr))
    if max_val > 0:
        arr = arr / max_val * 0.95
    return arr

# 1. Populate Target Classes
for folder_name, phrase, lang in TARGET_CONFIG:
    dst_dir = f"{MICRO_DIR}/dataset/{folder_name}"
    existing_samples = glob.glob(f"{dst_dir}/*.wav")
    
    if len(existing_samples) >= 15:
        print(f"Found {len(existing_samples)} existing samples for '{phrase}'. Skipping synthesis.")
        continue
        
    print(f"Generating acoustic samples and variations for '{phrase}' ({lang})...")
    tts = gTTS(text=phrase, lang=lang, slow=False)
    mp3_fp = io.BytesIO()
    tts.write_to_fp(mp3_fp)
    mp3_fp.seek(0)
    base_audio, _ = librosa.load(mp3_fp, sr=16000)
    
    clip_count = len(existing_samples)
    pitch_shifts = [-2.5, -1.5, -0.5, 0.0, 0.5, 1.5, 2.5]
    time_stretches = [0.85, 0.92, 1.0, 1.08, 1.18]
    
    for pitch in pitch_shifts:
        shifted = librosa.effects.pitch_shift(base_audio, sr=16000, n_steps=pitch)
        for stretch in time_stretches:
            stretched = librosa.effects.time_stretch(shifted, rate=stretch)
            formatted = pad_or_trim(stretched)
            sf.write(f"{dst_dir}/clip_{clip_count:04d}.wav", formatted, 16000, subtype="PCM_16")
            clip_count += 1
            
            # Background noise variation
            noise = np.random.normal(0, np.random.uniform(0.005, 0.02), len(formatted))
            noisy = pad_or_trim(formatted + noise)
            sf.write(f"{dst_dir}/clip_{clip_count:04d}.wav", noisy, 16000, subtype="PCM_16")
            clip_count += 1
            
    print(f"-> Created {len(glob.glob(f'{dst_dir}/*.wav'))} samples for {phrase}")

# 2. Negative speech clips (_unknown_)
unknown_dir = f"{MICRO_DIR}/dataset/_unknown_"
if len(glob.glob(f"{unknown_dir}/*.wav")) < 150:
    print("\nCollecting negative conversational clips (_unknown_)...")
    # 100% token-free local negative vocabulary generation
    # 100% token-free local negative vocabulary generation
        print("Generating negative conversational vocabulary...")
        neg_words = ["habari", "jambo", "nairobi", "sawa", "asante", "rafiki", "simu", "daktari", "shule", "pesa", "safari", "barabara", "hello", "yes", "no", "stop", "good", "friend", "water", "today"]
        idx = 0
        for word in neg_words:
            tts = gTTS(text=word, lang="sw" if idx % 2 == 0 else "en", slow=False)
            mp3_fp = io.BytesIO()
            tts.write_to_fp(mp3_fp)
            mp3_fp.seek(0)
            aud, _ = librosa.load(mp3_fp, sr=16000)
            for rate in [0.85, 0.95, 1.0, 1.1, 1.2]:
                stretched = pad_or_trim(librosa.effects.time_stretch(aud, rate=rate))
                sf.write(f"{unknown_dir}/unk_{idx:04d}.wav", stretched, 16000, subtype="PCM_16")
                idx += 1
                noise = pad_or_trim(stretched + np.random.normal(0, 0.015, len(stretched)))
                sf.write(f"{unknown_dir}/unk_{idx:04d}.wav", noise, 16000, subtype="PCM_16")
                idx += 1

print(f"-> Total _unknown_ clips: {len(glob.glob(f'{unknown_dir}/*.wav'))}")

# 3. Ambient silence & room noise (_silence_)
silence_dir = f"{MICRO_DIR}/dataset/_silence_"
print("Generating background silence and ambient noise...")
for i in range(150):
    noise_level = np.random.uniform(0.0001, 0.015)
    noise = np.random.normal(0, noise_level, 16000).astype(np.float32)
    sf.write(f"{silence_dir}/silence_{i:04d}.wav", noise, 16000, subtype="PCM_16")
print(f"-> Total _silence_ clips: {len(glob.glob(f'{silence_dir}/*.wav'))}")

In [ ]:
# ===========================================================
# CELL 4 — Audio Preprocessing & Spectrogram Extraction
# ===========================================================
import tensorflow as tf
import numpy as np

SAMPLE_RATE = 16000
DURATION_SAMPLES = 16000
NUM_CLASSES = len(CLASSES)

def get_spectrogram(waveform):
    waveform = tf.pad(waveform, [[0, tf.maximum(0, DURATION_SAMPLES - tf.shape(waveform)[0])]])
    waveform = waveform[:DURATION_SAMPLES]
    
    spectrogram = tf.signal.stft(
        waveform,
        frame_length=640,
        frame_step=320,
        fft_length=1024
    )
    spectrogram = tf.abs(spectrogram)
    
    num_spectrogram_bins = tf.shape(spectrogram)[-1]
    linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=40,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=80.0,
        upper_edge_hertz=7600.0
    )
    mel_spectrogram = tf.tensordot(spectrogram, linear_to_mel_weight_matrix, 1)
    mel_spectrogram.set_shape(spectrogram.shape[:-1].concatenate(linear_to_mel_weight_matrix.shape[-1:]))
    
    log_mel = tf.math.log(mel_spectrogram + 1e-6)
    return tf.expand_dims(log_mel, -1)

def load_wav_file(file_path):
    audio_binary = tf.io.read_file(file_path)
    audio, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)
    return tf.squeeze(audio, axis=-1)

data_dir = f'{MICRO_DIR}/dataset'
file_list = []
label_list = []
for label_idx, class_name in enumerate(CLASSES):
    wavs = glob.glob(f'{data_dir}/{class_name}/*.wav')
    file_list.extend(wavs)
    label_list.extend([label_idx] * len(wavs))

print(f'Total dataset: {len(file_list)} audio clips across {NUM_CLASSES} classes')

ds = tf.data.Dataset.from_tensor_slices((file_list, label_list))
ds = ds.shuffle(buffer_size=len(file_list), seed=42)

def map_fn(path, label):
    audio = load_wav_file(path)
    spec = get_spectrogram(audio)
    return spec, label

ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)

val_size = int(len(file_list) * 0.15)
train_ds = ds.skip(val_size).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = ds.take(val_size).batch(32).prefetch(tf.data.AUTOTUNE)

for sample_x, sample_y in train_ds.take(1):
    print(f'Input Spectrogram Shape: {sample_x.shape}')
    input_shape = sample_x.shape[1:]

In [ ]:
# ===========================================================
# CELL 5 — Define Ultra-Compact TinyML Model (Depthwise Separable CNN)
# ===========================================================
from tensorflow.keras import layers, models

def create_micro_kws_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.Conv2D(16, (3, 3), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        
        layers.DepthwiseConv2D((3, 3), strides=(1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(32, (1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        
        layers.DepthwiseConv2D((3, 3), strides=(1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(48, (1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

micro_model = create_micro_kws_model(input_shape, NUM_CLASSES)
micro_model.summary()

micro_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ===========================================================
# CELL 6 — Train Micro KWS Model
# ===========================================================
EPOCHS = 35

history = micro_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
    ]
)

loss, acc = micro_model.evaluate(val_ds)
print(f'\n✅ Validation Accuracy: {acc * 100:.2f}%')

In [ ]:
# ===========================================================
# CELL 8 — Export C Header `safeguard_micro_model.h` for ESP32-C3
# ===========================================================
header_path = f"{MICRO_DIR}/safeguard_micro_model.h"

with open(tflite_path, "rb") as f:
    bytes_data = f.read()

lines = [
    "#ifndef SAFEGUARD_MICRO_MODEL_H",
    "#define SAFEGUARD_MICRO_MODEL_H",
    "",
    "#include <stdint.h>",
    "",
    "// SafeGuard TinyML Keyword Spotting Model for ESP32-C3",
    f"// Model size: {len(bytes_data)} bytes ({len(bytes_data)/1024:.2f} KB)",
    "// Classes: 0=_silence_, 1=_unknown_, 2=tuma_msaada, 3=piga_polisi, 4=send_help, 5=call_police",
    "",
    f"const unsigned int g_safeguard_model_len = {len(bytes_data)};",
    "alignas(16) const uint8_t g_safeguard_model_data[] = {"
]

for i in range(0, len(bytes_data), 12):
    chunk = bytes_data[i:i+12]
    lines.append("  " + ", ".join([f"0x{b:02x}" for b in chunk]) + ",")

lines.append("};")
lines.append("")
lines.append("#endif // SAFEGUARD_MICRO_MODEL_H\n")

with open(header_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"🎉 Complete! C Header ready for ESP32: {header_path}")
print("Size:", len(bytes_data), "bytes")

# Trigger browser download in Google Colab
try:
    from google.colab import files
    files.download(header_path)
    print("⚡ Download dialog triggered in your browser!")
except Exception as e:
    print(f"To download manually, right click on {header_path} in the Colab file explorer on the left and select Download.")


In [ ]:
# ===========================================================
# CELL 8 — Export C Header `safeguard_micro_model.h` for ESP32-C3
# ===========================================================
header_path = f'{MICRO_DIR}/safeguard_micro_model.h'

with open(tflite_path, 'rb') as f:
    bytes_data = f.read()

lines = [
    '#ifndef SAFEGUARD_MICRO_MODEL_H',
    '#define SAFEGUARD_MICRO_MODEL_H',
    '',
    '#include <stdint.h>',
    '',
    '// SafeGuard TinyML Keyword Spotting Model for ESP32-C3',
    f'// Model size: {len(bytes_data)} bytes ({len(bytes_data)/1024:.2f} KB)',
    f'// Classes: 0=_silence_, 1=_unknown_, 2=tuma_msaada, 3=piga_polisi, 4=send_help, 5=call_police',
    '',
    f'const unsigned int g_safeguard_model_len = {len(bytes_data)};',
    'alignas(16) const uint8_t g_safeguard_model_data[] = {'
]

for i in range(0, len(bytes_data), 12):
    chunk = bytes_data[i:i+12]
    lines.append('  ' + ', '.join([f'0x{b:02x}' for b in chunk]) + ',')

lines.append('};')
lines.append('')
lines.append('#endif // SAFEGUARD_MICRO_MODEL_H\n')

with open(header_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'🎉 Complete! C Header ready for ESP32: {header_path}')
print('Download this file and place it in your firmware/gps_safety_pendant/ folder!')